### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_csv('R:/PROJECTS/User Journey Funnel Analysis/user_journey_funnel_analysis/retail_user_behavior_100k.csv')
print('Datatypes of each column:')
df.info()
df.head()
print(df['user_action'].value_counts())
print('Overall conversion rate at row level: ',df['is_conversion'].mean())

Datatypes of each column:
<class 'pandas.DataFrame'>
RangeIndex: 108584 entries, 0 to 108583
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   session_id         108584 non-null  str    
 1   user_id            108584 non-null  str    
 2   timestamp_utc      108584 non-null  str    
 3   event_index        108584 non-null  int64  
 4   user_action        108584 non-null  str    
 5   product_id         108584 non-null  str    
 6   category           108584 non-null  str    
 7   brand              108584 non-null  str    
 8   price              108584 non-null  float64
 9   channel            108584 non-null  str    
 10  device_type        108584 non-null  str    
 11  region             108584 non-null  str    
 12  traffic_source     108584 non-null  str    
 13  time_spent_sec     108584 non-null  int64  
 14  session_length     108584 non-null  int64  
 15  interaction_count  108584 non-null  

In [18]:
df.shape

(108584, 18)

In [10]:
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])

In [11]:
# Checking for missing values
print('Missing values in each column:', df.isnull().sum())

Missing values in each column: session_id           0
user_id              0
timestamp_utc        0
event_index          0
user_action          0
product_id           0
category             0
brand                0
price                0
channel              0
device_type          0
region               0
traffic_source       0
time_spent_sec       0
session_length       0
interaction_count    0
is_conversion        0
drop_off_flag        0
dtype: int64


In [12]:
# Checking for duplicates
print('Number of duplicate rows:', df.duplicated().sum())

Number of duplicate rows: 0


In [14]:
# Number of unique sessions
print('Number of unique sessions:', df['session_id'].nunique())

# Number of unique users
print('Number of unique users:', df['user_id'].nunique())

Number of unique sessions: 18000
Number of unique users: 6806


In [19]:
# Does is_conversion=1 always align with a 'purchase' action row?
print('is_conversion=1 rows by user_action:')
print(df[df.is_conversion==1].user_action.value_counts())
print()
print('drop_off_flag=1 rows by user_action:')
print(df[df.drop_off_flag==1].user_action.value_counts())
print()
# session-level: does every session have exactly one is_conversion value (i.e. is it session-level truth repeated on every row)?
print('Sessions where is_conversion varies within session:', (df.groupby('session_id').is_conversion.nunique() > 1).sum())

is_conversion=1 rows by user_action:
user_action
purchase    4203
Name: count, dtype: int64

drop_off_flag=1 rows by user_action:
user_action
drop    13797
Name: count, dtype: int64

Sessions where is_conversion varies within session: 4203


In [15]:
session_df = df.groupby('session_id').agg(
    user_id=('user_id', 'first'),
    channel=('channel', 'first'),
    device_type=('device_type', 'first'),
    region=('region', 'first'),
    traffic_source=('traffic_source', 'first'),
    category=('category', 'first'),
    session_length=('session_length', 'first'),
    total_time_spent=('time_spent_sec', 'sum'),
    total_interactions=('interaction_count', 'max'),
    num_events=('event_index', 'max'),
    converted=('is_conversion', 'max'),
    dropped_off=('drop_off_flag', 'max'),
    purchase_value=('price', lambda x: x[df.loc[x.index, 'user_action'] == 'purchase'].sum())
).reset_index()

print(session_df.shape)
session_df.head()
print('Session-level conversion rate:', session_df.converted.mean())

(18000, 14)
Session-level conversion rate: 0.2335


In [20]:
print(session_df.shape)
print(session_df.head().to_string())
print()
print('Session-level conversion rate:', round(session_df.converted.mean()*100, 2), '%')
print('Row-level conversion rate (old, wrong denominator):', round(df.is_conversion.mean()*100, 2), '%')
session_df.to_csv('session_level.csv', index=False)

(18000, 14)
  session_id  user_id channel device_type region traffic_source     category  session_length  total_time_spent  total_interactions  num_events  converted  dropped_off  purchase_value
0   S0000001  U000372  mobile     desktop     JP        organic  Electronics               4                74                   4           4          0            1            0.00
1   S0000002  U004812  mobile      tablet     UK         direct    Groceries               4                60                   4           4          0            1            0.00
2   S0000003  U001935     web     android     AU        organic    Groceries               6               122                   6           6          1            0          261.82
3   S0000004  U001996     app     desktop     IN      affiliate       Sports               3                60                   3           3          0            1            0.00
4   S0000005  U000024     web     android     CA      affiliate  Accessor

In [25]:
print('Sessions with BOTH converted=1 and dropped_off=1:', ((session_df.converted==1) & (session_df.dropped_off==1)).sum())
print('Sessions with NEITHER (converted=0 and dropped_off=0):', ((session_df.converted==0) & (session_df.dropped_off==0)).sum())
print(session_df[['converted','dropped_off']].value_counts())

Sessions with BOTH converted=1 and dropped_off=1: 0
Sessions with NEITHER (converted=0 and dropped_off=0): 0
converted  dropped_off
0          1              13797
1          0               4203
Name: count, dtype: int64


In [26]:
# What's the typical order of actions within a session?
print(df.groupby('event_index')['user_action'].value_counts(normalize=True).unstack().round(3))

user_action  add_to_cart  click   drop  purchase   view  wishlist
event_index                                                      
1                    NaN    NaN    NaN       NaN  1.000       NaN
2                  0.154  0.382    NaN       NaN  0.364     0.100
3                  0.150  0.345  0.067     0.020  0.333     0.085
4                  0.138  0.325  0.123     0.040  0.295     0.079
5                  0.119  0.284  0.197     0.059  0.267     0.073
6                  0.107  0.249  0.265     0.078  0.240     0.061
7                  0.091  0.221  0.322     0.102  0.210     0.053
8                  0.085  0.196  0.372     0.108  0.186     0.052
9                  0.071  0.186  0.401     0.127  0.174     0.041
10                 0.057  0.152  0.473     0.126  0.155     0.037
11                 0.050  0.150  0.463     0.150  0.132     0.055
12                 0.054  0.150  0.435     0.116  0.197     0.048
13                   NaN    NaN  0.727     0.273    NaN       NaN


In [31]:
stages = ['view', 'click', 'add_to_cart', 'purchase']
funnel_counts = {}
for stage in stages:
    sessions_reaching_stage = df[df['user_action'] == stage]['session_id'].nunique()
    funnel_counts[stage] = sessions_reaching_stage

funnel_df = pd.Series(funnel_counts)
print(funnel_df)
print()
print('Drop-off between stages (%):')
print((1 - funnel_df / funnel_df.shift(1)).round(2).fillna(1) * 100)

view           18000
click          14116
add_to_cart     8833
purchase        4203
dtype: int64

Drop-off between stages (%):
view           100.0
click           22.0
add_to_cart     37.0
purchase        52.0
dtype: float64


In [32]:
# Conversion rate by channel
print(session_df.groupby('channel')['converted'].mean().sort_values(ascending=False))

# Conversion rate by device_type
print(session_df.groupby('device_type')['converted'].mean().sort_values(ascending=False))

channel
app       0.267428
mobile    0.223675
web       0.209755
Name: converted, dtype: float64
device_type
android    0.245625
ios        0.239351
desktop    0.229134
tablet     0.219485
Name: converted, dtype: float64
